In [7]:
import dtlpy as dl

In [ ]:
dl.setenv('rc')
dl.logout()
dl.login()

In [8]:
project = dl.projects.get(project_id="21bc0e73-dbc7-422c-b28a-a41fecf60a72")

In [ ]:
# dpk = dl.dpks.get(dpk_name="image-preprocess-husam")
# dpk.delete()

In [10]:
dpk = project.dpks.publish()
# Install or update the application
try:
    app = project.apps.get(app_name=dpk.display_name)
    app.dpk_version = dpk.version
    app.update()
except dl.exceptions.NotFound as e:
    print("error = ", e)
    print("installing ...")
    app = project.apps.install(dpk=dpk)

In [ ]:
dataset = dl.datasets.get(dataset_id="6a296c76e54b4f3fc10e210f")

# dataset.open_in_web()
# item = dataset.items.upload(local_path=r"C:\Users\Husam_Massalha\Documents\git_repos\image-preprocess\artifacts\Screenshot 2026-05-02 142513.png")

In [ ]:
import copy
item = dl.items.get(item_id=item.id)
# item = dl.items.get(item_id="6a296ca003c1ed8d7aba9c05")
original_metadata = copy.deepcopy(item.metadata)

In [ ]:

thumbnail_item = dl.items.get(item_id=original_metadata["system"]["thumbnailId"])

In [ ]:
thumbnail_item.download(local_path="original_thumbnail")

In [ ]:
thumbnail_item.delete()

In [ ]:
item.metadata

In [ ]:
# Remove only the fields added by the preprocessing service
# From system: width, height, channels
for key in ["width", "height", "channels"]:
    item.metadata.get("system", {}).pop(key, None)

# From top-level metadata: exif, location
for key in ["exif", "location"]:
    item.metadata.pop(key, None)

item.update(system_metadata=True)

In [12]:
new_service = dl.services.get(service_id="6a296af41dcd47a2b3f6d857")

In [ ]:
service_execution = new_service.execute(
        execution_input={"item": item.id},
        project_id=project.id,
        function_name="on_create",
    )
execution = service_execution.wait()

In [ ]:
buffer = item.download(save_locally=False)


In [ ]:
item = dl.items.get(item_id=item.id)
new_metadata = item.metadata

In [13]:
import json

def compare_metadata(old, new, path=""):
    """Recursively compare two metadata dicts and print differences."""
    all_keys = set(list(old.keys()) + list(new.keys())) if isinstance(old, dict) and isinstance(new, dict) else set()

    if not isinstance(old, dict) or not isinstance(new, dict):
        if old != new:
            print(f"  CHANGED  {path}: {old!r} -> {new!r}")
        return

    for key in sorted(all_keys):
        full_path = f"{path}.{key}" if path else key
        if key not in old:
            val = new[key]
            print(f"  ADDED    {full_path}: {json.dumps(val, indent=2) if isinstance(val, dict) else val!r}")
        elif key not in new:
            val = old[key]
            print(f"  REMOVED  {full_path}: {json.dumps(val, indent=2) if isinstance(val, dict) else val!r}")
        else:
            compare_metadata(old[key], new[key], full_path)

# print("=" * 60)
# print("Metadata comparison: original vs. new (after preprocessing)")
# print("=" * 60)
# compare_metadata(original_metadata, new_metadata)
# print("=" * 60)

In [ ]:
new_thumb_item = dl.items.get(item_id="6a29c57203c1ed8d7abb3d87")
new_thumb_item.download(local_path="new_thumbnail/")

In [ ]:
for item in dataset.items.list().all():
    item.delete()

In [ ]:
import os, copy, time, hashlib, json

ASSETS_DIR = r"C:\Users\Husam_Massalha\Documents\git_repos\image-preprocess\artifacts"
WAIT_MINUTES = 5

# Metadata keys set by this repo's preprocessing service (for comparison)
# NOTE: thumbnailId excluded — always a new ID; validated separately
# NOTE: quality_scores excluded — set by default platform, not this service
SERVICE_SYSTEM_KEYS = ["width", "height", "channels", "exif", "location"]
SERVICE_TOP_KEYS = ["exif", "location"]
SERVICE_USER_KEYS = ["location"]

def sha256_of_item(item_obj):
    """Download an item into memory and return its SHA-256 hex digest."""
    buf = item_obj.download(save_locally=False)
    if hasattr(buf, "read"):
        data = buf.read()
    else:
        data = buf
    return hashlib.sha256(data).hexdigest()

def compare_service_metadata(old_meta, new_meta):
    """Compare only the metadata fields that the preprocessing service creates."""
    diffs = []

    old_sys = old_meta.get("system", {})
    new_sys = new_meta.get("system", {})

    for key in SERVICE_SYSTEM_KEYS:
        old_val = old_sys.get(key)
        new_val = new_sys.get(key)
        if old_val != new_val:
            if isinstance(old_val, dict) and isinstance(new_val, dict):
                for sub in sorted(set(list(old_val.keys()) + list(new_val.keys()))):
                    ov = old_val.get(sub)
                    nv = new_val.get(sub)
                    if ov != nv:
                        diffs.append(f"  system.{key}.{sub}: {ov!r} -> {nv!r}")
            else:
                diffs.append(f"  system.{key}: {old_val!r} -> {new_val!r}")

    old_user = old_meta.get("user", {})
    new_user = new_meta.get("user", {})
    for key in SERVICE_USER_KEYS:
        old_val = old_user.get(key)
        new_val = new_user.get(key)
        if old_val != new_val:
            diffs.append(f"  user.{key}: {json.dumps(old_val, indent=2) if isinstance(old_val, dict) else old_val!r} -> {json.dumps(new_val, indent=2) if isinstance(new_val, dict) else new_val!r}")

    return diffs

print("Setup done.")

Setup done.


In [15]:
# Step 1: Clean the dataset and upload all assets
print("Cleaning dataset...")
for existing_item in dataset.items.list().all():
    existing_item.delete()

uploaded_items = dataset.items.upload(local_path=ASSETS_DIR)
if not isinstance(uploaded_items, list):
    uploaded_items = [uploaded_items]
print(f"Uploaded {len(uploaded_items)} items from {ASSETS_DIR}")

# Step 2: Wait for the DEFAULT platform preprocessing to finish
print(f"Waiting {WAIT_MINUTES} minutes for default platform preprocessing...")
time.sleep(WAIT_MINUTES * 60)
print("Done waiting.")

Cleaning dataset...
Uploaded 1 items from C:\Users\Husam_Massalha\Documents\git_repos\image-preprocess\assets
Waiting 5 minutes for default platform preprocessing...
Done waiting.


In [16]:
# Step 3: Snapshot original (default) metadata and thumbnail for every item
from pathlib import Path
from PIL import Image
from io import BytesIO

DEFAULT_THUMB_DIR = Path("thumbnails/default")
CUSTOM_THUMB_DIR = Path("thumbnails/custom")
DEFAULT_THUMB_DIR.mkdir(parents=True, exist_ok=True)
CUSTOM_THUMB_DIR.mkdir(parents=True, exist_ok=True)

originals = {}  # item_id -> {"metadata": ..., "thumb_hash": ..., "thumb_size": ...}

for page in dataset.items.list():
    for item in page:
        item = dl.items.get(item_id=item.id)  # refresh
        meta_snapshot = copy.deepcopy(item.metadata)
        thumb_hash = None
        thumb_size = None
        thumb_id = meta_snapshot.get("system", {}).get("thumbnailId")
        if thumb_id:
            try:
                thumb_item = dl.items.get(item_id=thumb_id)
                buf = thumb_item.download(save_locally=False)
                data = buf.read() if hasattr(buf, "read") else buf
                thumb_hash = hashlib.sha256(data).hexdigest()
                # Save locally
                save_name = f"{Path(item.name).stem}_default.png"
                save_path = DEFAULT_THUMB_DIR / save_name
                with open(save_path, "wb") as f:
                    f.write(data)
                # Get dimensions
                img = Image.open(BytesIO(data))
                thumb_size = img.size  # (width, height)
                print(f"  Saved snapshot for {item.name} (thumb: {thumb_size[0]}x{thumb_size[1]})")
            except Exception as e:
                print(f"  Warning: could not download thumbnail for {item.name}: {e}")
        else:
            print(f"  Saved snapshot for {item.name} (no default thumbnail)")
        originals[item.id] = {
            "name": item.name,
            "metadata": meta_snapshot,
            "thumb_hash": thumb_hash,
            "thumb_size": thumb_size,
        }

print(f"\nSnapshots saved for {len(originals)} items.")
print(f"Default thumbnails saved to: {DEFAULT_THUMB_DIR.resolve()}")

  Saved snapshot for 1.jpg (no default thumbnail)
  Saved snapshot for 0000000162.jpg (no default thumbnail)
  Saved snapshot for 0000000162.png (no default thumbnail)
  Saved snapshot for 1.png (no default thumbnail)
  Saved snapshot for 2.jpg (no default thumbnail)
  Saved snapshot for DogeJPEG_3.jpeg (no default thumbnail)
  Saved snapshot for 4.jpg (no default thumbnail)
  Saved snapshot for DogeJPEG.jpeg (no default thumbnail)
  Saved snapshot for DogeJPEG_1.jpeg (no default thumbnail)
  Saved snapshot for DogeJPEG_2.jpeg (no default thumbnail)
  Saved snapshot for anchors.webp (thumb: 128x64)
  Saved snapshot for button.gif (thumb: 78x53)
  Saved snapshot for bmp_image_item.bmp (thumb: 122x128)
  Saved snapshot for HTm48iG_1.jpeg (no default thumbnail)
  Saved snapshot for FByIhxTWUAIapK5.jpg (no default thumbnail)
  Saved snapshot for HTm48iG.jpeg (no default thumbnail)
  Saved snapshot for HTm48iG_2.jpeg (no default thumbnail)
  Saved snapshot for image_332.jpg (no default thum

In [17]:
# Step 4: Strip service-created metadata and delete thumbnails
for item_id, info in originals.items():
    item = dl.items.get(item_id=item_id)

    # Delete existing thumbnail
    thumb_id = item.metadata.get("system", {}).get("thumbnailId")
    if thumb_id:
        try:
            dl.items.get(item_id=thumb_id).delete()
        except Exception:
            pass

    # Remove service-created system fields
    sys_meta = item.metadata.get("system", {})
    for key in SERVICE_SYSTEM_KEYS:
        sys_meta.pop(key, None)

    # Remove service-created top-level metadata keys
    for key in SERVICE_TOP_KEYS:
        item.metadata.pop(key, None)

    # Remove service-created user keys
    user_meta = item.metadata.get("user", {})
    for key in SERVICE_USER_KEYS:
        user_meta.pop(key, None)

    item.update(system_metadata=True)
    print(f"  Stripped: {info['name']}")

print("All items stripped.")

  Stripped: 1.jpg
  Stripped: 0000000162.jpg
  Stripped: 0000000162.png
  Stripped: 1.png
  Stripped: 2.jpg
  Stripped: DogeJPEG_3.jpeg
  Stripped: 4.jpg
  Stripped: DogeJPEG.jpeg
  Stripped: DogeJPEG_1.jpeg
  Stripped: DogeJPEG_2.jpeg
  Stripped: anchors.webp
  Stripped: button.gif
  Stripped: bmp_image_item.bmp
  Stripped: HTm48iG_1.jpeg
  Stripped: FByIhxTWUAIapK5.jpg
  Stripped: HTm48iG.jpeg
  Stripped: HTm48iG_2.jpeg
  Stripped: image_332.jpg
  Stripped: faledetl.jpg
  Stripped: image_425.jpg
  Stripped: jfif_image_item.jfif
  Stripped: mask_should_be.png
  Stripped: Screenshot 2026-05-02 142513.png
  Stripped: label1.png
  Stripped: iss634.webp
  Stripped: rgb24pal.bmp
  Stripped: test.jpg
  Stripped: smile.gif
  Stripped: instance_should_be.png
  Stripped: pal1wb.bmp
  Stripped: webp_image_item.webp
All items stripped.


In [18]:
# Step 5: Execute the custom service on every item and wait for completion
executions = []
for item_id, info in originals.items():
    ex = new_service.execute(
        execution_input={"item": item_id},
        project_id=project.id,
        function_name="on_create",
    )
    executions.append((item_id, info["name"], ex))
    print(f"  Triggered execution for {info['name']}")

print(f"\nWaiting for {len(executions)} executions to finish...")
for item_id, name, ex in executions:
    try:
        ex.wait()
        print(f"  Done: {name}")
    except Exception as e:
        print(f"  FAILED: {name} — {e}")

print("All executions finished.")

  Triggered execution for 1.jpg
  Triggered execution for 0000000162.jpg
  Triggered execution for 0000000162.png
  Triggered execution for 1.png
  Triggered execution for 2.jpg
  Triggered execution for DogeJPEG_3.jpeg
  Triggered execution for 4.jpg
  Triggered execution for DogeJPEG.jpeg
  Triggered execution for DogeJPEG_1.jpeg
  Triggered execution for DogeJPEG_2.jpeg
  Triggered execution for anchors.webp
  Triggered execution for button.gif
  Triggered execution for bmp_image_item.bmp
  Triggered execution for HTm48iG_1.jpeg
  Triggered execution for FByIhxTWUAIapK5.jpg
  Triggered execution for HTm48iG.jpeg
  Triggered execution for HTm48iG_2.jpeg
  Triggered execution for image_332.jpg
  Triggered execution for faledetl.jpg
  Triggered execution for image_425.jpg
  Triggered execution for jfif_image_item.jfif
  Triggered execution for mask_should_be.png
  Triggered execution for Screenshot 2026-05-02 142513.png
  Triggered execution for label1.png
  Triggered execution for iss

In [19]:
# Step 6: Compare custom service metadata & thumbnail vs default (original)
print("=" * 80)
print("COMPARISON: Default Platform vs Custom Service")
print("=" * 80)

pass_count = 0
fail_count = 0
missing_thumb_id = []

for item_id, info in originals.items():
    item = dl.items.get(item_id=item_id)
    new_meta = item.metadata
    old_meta = info["metadata"]

    print(f"\n{'='*80}")
    print(f"{info['name']} ({item_id}):")

    # --- Validate thumbnailId is present (not comparing old vs new) ---
    new_thumb_id = new_meta.get("system", {}).get("thumbnailId")
    if new_thumb_id:
        print(f"  THUMBNAIL ID: present ({new_thumb_id})")
    else:
        print(f"  THUMBNAIL ID: MISSING")
        missing_thumb_id.append(info["name"])

    # --- Metadata comparison (service-created fields only) ---
    diffs = compare_service_metadata(old_meta, new_meta)
    if diffs:
        print("  METADATA DIFFS (service fields):")
        for d in diffs:
            print(f"    {d}")
        fail_count += 1
    else:
        print("  METADATA: match")
        pass_count += 1

    # --- Thumbnail content & dimensions comparison ---
    old_thumb_hash = info["thumb_hash"]
    old_thumb_size = info["thumb_size"]
    new_thumb_hash = None
    new_thumb_size = None

    if new_thumb_id:
        try:
            new_thumb_item = dl.items.get(item_id=new_thumb_id)
            buf = new_thumb_item.download(save_locally=False)
            data = buf.read() if hasattr(buf, "read") else buf
            new_thumb_hash = hashlib.sha256(data).hexdigest()
            # Save locally
            save_name = f"{Path(info['name']).stem}_custom.png"
            save_path = CUSTOM_THUMB_DIR / save_name
            with open(save_path, "wb") as f:
                f.write(data)
            # Get dimensions
            img = Image.open(BytesIO(data))
            new_thumb_size = img.size
        except Exception as e:
            print(f"  THUMBNAIL: could not download -- {e}")

    # Hash comparison
    if old_thumb_hash and new_thumb_hash:
        if old_thumb_hash == new_thumb_hash:
            print("  THUMBNAIL CONTENT: identical")
        else:
            print(f"  THUMBNAIL CONTENT: DIFFER  (default={old_thumb_hash[:16]}... vs custom={new_thumb_hash[:16]}...)")
    elif old_thumb_hash is None and new_thumb_hash is None:
        print("  THUMBNAIL CONTENT: both absent")
    else:
        print(f"  THUMBNAIL CONTENT: default={'present' if old_thumb_hash else 'absent'}, custom={'present' if new_thumb_hash else 'absent'}")

    # Dimensions comparison
    if old_thumb_size and new_thumb_size:
        if old_thumb_size == new_thumb_size:
            print(f"  THUMBNAIL DIMS:    identical ({old_thumb_size[0]}x{old_thumb_size[1]})")
        else:
            print(f"  THUMBNAIL DIMS:    DIFFER  (default={old_thumb_size[0]}x{old_thumb_size[1]} vs custom={new_thumb_size[0]}x{new_thumb_size[1]})")
    elif old_thumb_size is None and new_thumb_size is None:
        print("  THUMBNAIL DIMS:    both absent")
    else:
        old_str = f"{old_thumb_size[0]}x{old_thumb_size[1]}" if old_thumb_size else "absent"
        new_str = f"{new_thumb_size[0]}x{new_thumb_size[1]}" if new_thumb_size else "absent"
        print(f"  THUMBNAIL DIMS:    default={old_str}, custom={new_str}")

print(f"\n{'='*80}")
print(f"SUMMARY: {pass_count} matching, {fail_count} with metadata diffs (out of {len(originals)} items)")
if missing_thumb_id:
    print(f"WARNING: {len(missing_thumb_id)} items missing thumbnailId: {missing_thumb_id}")
else:
    print(f"All {len(originals)} items have valid thumbnailId")
print(f"Thumbnails saved to: {DEFAULT_THUMB_DIR.resolve()} and {CUSTOM_THUMB_DIR.resolve()}")
print("Done.")

COMPARISON: Default Platform vs Custom Service

1.jpg (6a2c2a586150bfd566a64287):
  THUMBNAIL ID: present (6a2c2c136150bfd566a64302)
  METADATA: match
  THUMBNAIL CONTENT: default=absent, custom=present
  THUMBNAIL DIMS:    default=absent, custom=128x128

0000000162.jpg (6a2c2a586150bfd566a64289):
  THUMBNAIL ID: present (6a2c2c139ae827ee1b375b47)
  METADATA: match
  THUMBNAIL CONTENT: default=absent, custom=present
  THUMBNAIL DIMS:    default=absent, custom=128x64

0000000162.png (6a2c2a58b7e0d620f4432fe1):
  THUMBNAIL ID: present (6a2c2c14ba22474c2d05a585)
  METADATA: match
  THUMBNAIL CONTENT: default=absent, custom=present
  THUMBNAIL DIMS:    default=absent, custom=128x64

1.png (6a2c2a58d36bdb8a60889761):
  THUMBNAIL ID: present (6a2c2c146150bfd566a64309)
  METADATA DIFFS (service fields):
      system.channels: 3 -> 1
  THUMBNAIL CONTENT: default=absent, custom=present
  THUMBNAIL DIMS:    default=absent, custom=85x128

2.jpg (6a2c2a58d36bdb8a60889763):
  THUMBNAIL ID: present 